In [1]:
!pip install xgboost lightgbm scikit-learn pandas numpy joblib matplotlib --quiet

import pandas as pd
import numpy as np
import gc
import joblib
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier

print("Imports successful")

Imports successful


In [2]:
data         = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-splits/splits.pkl")
X_train      = data["X_train"]
X_val        = data["X_val"]
X_test       = data["X_test"]
y_train      = data["y_train"]
y_val        = data["y_val"]
y_test       = data["y_test"]
del data
gc.collect()

le           = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-splits/label_encoder.pkl")
feature_cols = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-splits/feature_cols.pkl")
le_xgb       = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-splits/le_xgb.pkl")
rf_model     = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-splits/rf_model.pkl")
xgb_model    = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-splits/xgb_model.pkl")
svm_model    = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-splits/svm_model.pkl")

num_classes  = len(le.classes_)

print("All files loaded")
print(f"Train    : {X_train.shape[0]} rows")
print(f"Val      : {X_val.shape[0]} rows")
print(f"Test     : {X_test.shape[0]} rows")
print(f"Features : {X_train.shape[1]}")
print(f"Classes  : {num_classes}")

All files loaded
Train    : 34999 rows
Val      : 7500 rows
Test     : 7500 rows
Features : 1756
Classes  : 1117


In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

def get_oof_predictions(model, X, y, model_name, n_splits=3, label_encoder=None):
    skf       = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    n_classes = len(np.unique(y))
    oof       = np.zeros((X.shape[0], n_classes), dtype=np.float32)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"  [{model_name}] Fold {fold+1}/{n_splits} training...", end=" ")
        y_fold_train = y[tr_idx]
        y_fold_val   = y[val_idx]

        if model_name == "SVM":
            svm_base_fold = LinearSVC(
                max_iter     = 3000,
                class_weight = "balanced",
                random_state = 42,
                C            = 0.5
            )
            fold_model = CalibratedClassifierCV(estimator=svm_base_fold, cv=2)
            fold_model.fit(X[tr_idx], y_fold_train)
            oof[val_idx] = fold_model.predict_proba(X[val_idx])
            preds        = fold_model.predict(X[val_idx])

        elif label_encoder is not None:
            le_fold          = LabelEncoder()
            y_fold_train_fit = le_fold.fit_transform(y_fold_train)
            y_fold_val_enc   = le_fold.transform(y_fold_val)
            model.fit(
                X[tr_idx], y_fold_train_fit,
                eval_set = [(X[val_idx], y_fold_val_enc)],
                verbose  = False
            )
            oof[val_idx] = model.predict_proba(X[val_idx])
            preds        = le_fold.inverse_transform(model.predict(X[val_idx]))

        else:
            model.fit(X[tr_idx], y_fold_train)
            oof[val_idx] = model.predict_proba(X[val_idx])
            preds        = model.predict(X[val_idx])

        fold_acc = accuracy_score(y_fold_val, preds)
        print(f"fold acc: {fold_acc*100:.2f}%")

    print(f"  [{model_name}] OOF done\n")
    return oof

print("Generating OOF predictions ...\n")
print("--- Random Forest ---")
rf_oof  = get_oof_predictions(rf_model,  X_train, y_train, "RF",  n_splits=3)
print("--- SVM ---")
svm_oof = get_oof_predictions(svm_model, X_train, y_train, "SVM", n_splits=3)
print("--- XGBoost ---")
xgb_oof = get_oof_predictions(xgb_model, X_train, y_train, "XGBoost", n_splits=3,
                               label_encoder=le_xgb)

meta_train = np.hstack([rf_oof, xgb_oof, svm_oof])
del rf_oof, xgb_oof, svm_oof
gc.collect()

print(f"meta_train shape: {meta_train.shape}")
print(f" = {X_train.shape[0]} samples × "
      f"({num_classes} classes × 3 models = {num_classes*3} meta-features)")

Generating OOF predictions ...

--- Random Forest ---
  [RF] Fold 1/3 training... fold acc: 81.70%
  [RF] Fold 2/3 training... fold acc: 80.98%
  [RF] Fold 3/3 training... fold acc: 82.12%
  [RF] OOF done

--- SVM ---
  [SVM] Fold 1/3 training... fold acc: 81.91%
  [SVM] Fold 2/3 training... fold acc: 81.72%
  [SVM] Fold 3/3 training... fold acc: 82.66%
  [SVM] OOF done

--- XGBoost ---
  [XGBoost] Fold 1/3 training... fold acc: 67.46%
  [XGBoost] Fold 2/3 training... fold acc: 66.26%
  [XGBoost] Fold 3/3 training... fold acc: 66.96%
  [XGBoost] OOF done

meta_train shape: (34999, 1449)
 = 34999 samples × (1117 classes × 3 models = 3351 meta-features)


In [7]:
print("Retraining base models on full X_train...\n")

print("Fitting RF...")
rf_model.fit(X_train, y_train)
print("RF done")

print("Fitting SVM...")
svm_model.fit(X_train, y_train)
print("SVM done")

print("Fitting XGBoost...")
y_train_enc = le_xgb.transform(y_train)
y_val_enc   = le_xgb.transform(y_val)
xgb_model.fit(X_train, y_train_enc,                    
              eval_set=[(X_val, y_val_enc)],
             verbose = 100)
print("XGBoost done")

# Overwrite .pkl files with fully retrained versions
joblib.dump(rf_model,  "/kaggle/working/rf_model.pkl")
joblib.dump(xgb_model, "/kaggle/working/xgb_model.pkl")
joblib.dump(svm_model, "/kaggle/working/svm_model.pkl")

print("\nFully retrained base models saved")

Retraining base models on full X_train...

Fitting RF...
RF done
Fitting SVM...
SVM done
Fitting XGBoost...
[0]	validation_0-mlogloss:5.62838
[100]	validation_0-mlogloss:1.87928
[200]	validation_0-mlogloss:1.41690
[300]	validation_0-mlogloss:1.29461
[400]	validation_0-mlogloss:1.24770
[499]	validation_0-mlogloss:1.22558
XGBoost done

Fully retrained base models saved
